<a href="https://colab.research.google.com/github/saloninakat/retail-data-cleaning/blob/main/MiniProjectForEXL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#users.csv cleaning
import pandas as pd

users = pd.read_csv("users.csv")

users = users.drop_duplicates(subset=['user_id'])

users = users.dropna(subset=['user_id'])
users['name'] = users['name'].fillna("Unknown")
users['city'] = users['city'].fillna("Unknown")
users['name'] = users['name'].str.title().str.strip()
users['city'] = users['city'].str.title().str.strip()
users['name'] = users['name'].str.replace(r"[^A-Za-z\s]", "", regex=True)
users['city'] = users['city'].str.replace(r"[^A-Za-z\s]", "", regex=True)

date_cols = ['created_on', 'updated_on', 'dob']

for col in date_cols:
    if col in users.columns:
        users[col] = pd.to_datetime(users[col], errors='coerce')

users = users[users['user_id'].astype(str).str.strip() != ""]

try:
    users['user_id'] = users['user_id'].astype(int)
except:
    pass

users = users.sort_values(by='user_id')

users.to_csv("users_cleaned.csv", index=False)

users.head()

,user_id,name,city
3376,1,Unknown,Jodhpur
2654,2,Olivia Sharma,Pune
2245,3,Anaya Kapoor,Hyderabad
997,4,Unknown,Gwalior
1294,5,Unknown,Amritsar


In [ ]:
#product.csv cleaning

import pandas as pd
import re


df = pd.read_csv("products.csv")
df = df.drop_duplicates()
df['sku'] = df['sku'].astype(str).str.strip()
df['sku'] = df['sku'].str.upper().str.replace(" ", "")
df['sku'] = df['sku'].apply(lambda x: re.sub(r"[^A-Z0-9\-]", "", x))
df.loc[df['sku'].str.len() < 3, 'sku'] = "UNKNOWN_SKU"

df['product_name'] = (
    df['product_name']
    .astype(str)
    .str.strip()
    .str.lower()
)

category_map = {
    "electronics": ["mouse","keyboard","graphics","gpu","tv","television","smartwatch","watch","laptop","mobile","phone"],
    "appliances": ["microwave","fridge","refrigerator","ac","air conditioner","washing machine"],
    "furniture": ["sofa","table","chair","bed","cupboard"],
    "clothing": ["tshirt","shirt","pant","jeans","dress","sneaker","shoes","jacket"],
    "books": ["book","novel","fiction","story","author"],
    "home": ["decor","lamp","curtain","mat","home"],
    "kitchen":["cooker","mixer grinder"],
}

def assign_cat(name):
    for cat, keywords in category_map.items():
        for k in keywords:
            if k in name:
                return cat.capitalize()
    return "Others"

df['category'] = df['product_name'].apply(assign_cat)

df['product_name'] = df['product_name'].str.title()

if 'product_id' in df.columns:
    df = df.sort_values(by='product_id')
else:
    print("WARNING: No product_id column found for sorting.")

df.to_csv("products_cleaned.csv", index=False)

df.head(20)

,product_id,sku,product_name,category
381,1,SKU1,Mouse,Electronics
396,2,NAN,Watch,Electronics
357,3,SKU3,Graphics Card,Electronics
3,4,SKU-4,Sofa,Furniture
53,5,NAN,Television,Electronics
52,6,SKU6,Nan,Others
679,7,SKU7,Microwave,Appliances
28,8,SKU-8,Fiction Book,Books
257,9,SKU9,Sneakers,Clothing
704,10,SKU10,Smartwatch,Electronics


In [ ]:
#order.csv
import pandas as pd
import numpy as np
import re

orders = pd.read_csv("orders.csv")

orders = orders.drop_duplicates(subset=['order_id'])
orders = orders.dropna(subset=['order_id', 'user_id'])

orders['order_id'] = pd.to_numeric(orders['order_id'], errors='coerce')
orders['user_id'] = pd.to_numeric(orders['user_id'], errors='coerce')

orders = orders.dropna(subset=['order_id', 'user_id'])


orders['order_id'] = orders['order_id'].astype(int)
orders['user_id'] = orders['user_id'].astype(int)


orders['amount'] = (
    orders['amount']
    .astype(str)
    .str.replace(r'[^\d\.\-]', '', regex=True)
)

orders['amount'] = pd.to_numeric(orders['amount'], errors='coerce')


orders['amount'] = orders['amount'].apply(lambda x: abs(x) if pd.notnull(x) else np.nan)

orders = orders.dropna(subset=['amount'])


orders['order_timestamp'] = pd.to_datetime(orders['order_timestamp'], errors='coerce')


orders = orders.dropna(subset=['order_timestamp'])

orders.columns = orders.columns.str.strip()

orders = orders.sort_values(by=['order_timestamp'])

orders.to_csv("orders_cleaned.csv", index=False)

orders.head()

/tmp/ipython-input-3699779894.py:35: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  orders['order_timestamp'] = pd.to_datetime(orders['order_timestamp'], errors='coerce')


,order_id,user_id,order_timestamp,amount
16732,5463,4019,2022-01-02,1500.0
7561,19021,5891,2022-01-02,1500.0
1395,7565,5868,2022-01-03,1919.0
683,4760,3512,2022-01-05,4986.0
3709,11897,3408,2022-01-06,1500.0


In [ ]:
#order_item csv cleaning
import pandas as pd
import numpy as np
import re

order_items = pd.read_csv("order_items.csv")
original_data = order_items.copy()  # keep original for dirty report

order_items = order_items.drop_duplicates()


order_items['order_id'] = pd.to_numeric(order_items['order_id'], errors='coerce')
invalid_order_id = order_items[order_items['order_id'].isna()]
order_items = order_items[order_items['order_id'].notna()]
order_items['order_id'] = order_items['order_id'].astype(int)

order_items['sku'] = order_items['sku'].astype(str).str.strip().str.upper()
valid_sku_pattern = r'^[A-Z0-9_-]+$'
invalid_sku = order_items[~order_items['sku'].str.match(valid_sku_pattern)]
order_items = order_items[order_items['sku'].str.match(valid_sku_pattern)]

order_items['price'] = order_items['price'].astype(str).str.replace(r'[^\d\.\-]', '', regex=True)
order_items['price'] = pd.to_numeric(order_items['price'], errors='coerce')
invalid_price = order_items[order_items['price'].isna() | (order_items['price'] <= 0)]
order_items = order_items[order_items['price'] > 0]

order_items['quantity'] = order_items['quantity'].astype(str).str.replace(r'[^\d\-]', '', regex=True)
order_items['quantity'] = pd.to_numeric(order_items['quantity'], errors='coerce')
invalid_quantity = order_items[order_items['quantity'].isna() | (order_items['quantity'] <= 0)]
order_items = order_items[order_items['quantity'] > 0]

categorical_cols = ['status', 'category', 'payment_type']
invalid_cats = pd.DataFrame()
for col in categorical_cols:
    if col in order_items.columns:
        invalid_rows = order_items[order_items[col].isna() | (order_items[col].str.strip() == "")]
        invalid_cats = pd.concat([invalid_cats, invalid_rows])
        order_items = order_items[order_items[col].notna()]
        order_items[col] = order_items[col].str.strip().str.upper()

date_cols = [col for col in order_items.columns if 'date' in col.lower()]
invalid_dates = pd.DataFrame()
for col in date_cols:
    order_items[col] = pd.to_datetime(order_items[col], errors='coerce')
    invalid_dates = pd.concat([invalid_dates, order_items[order_items[col].isna()]])

duplicates_after_cleaning = order_items[order_items.duplicated(subset=['order_id', 'sku'], keep=False)]
order_items = order_items.drop_duplicates(subset=['order_id', 'sku'], keep='first')
order_items= order_items.sort_values(by=['order_id'])
order_items = order_items.reset_index(drop=True)

dirty_report = pd.concat([
    invalid_order_id,
    invalid_sku,
    invalid_price,
    invalid_quantity,
    duplicates_after_cleaning,
    invalid_dates,
    invalid_cats
]).drop_duplicates().reset_index(drop=True)

order_items.to_csv("order_items_cleaned.csv", index=False)
dirty_report.to_csv("order_items_dirty_report.csv", index=False)

print("Cleaned data2 preview:")
print(order_items.head())

print("\nDirty items removed preview:")
print(dirty_report.head())

Cleaned data2 preview:
   order_id  order_item_id       sku    price  quantity
0         3              3    SKU956  3149.00       2.0
1         3              1   SKU-258   100.00       2.0
2         3              2   SKU-905   999.99       2.0
3         4              2  SKU_1028  1649.00       2.0
4         5              3   SKU1146  1562.00       2.0

Dirty items removed preview:
   order_id  order_item_id      sku  price quantity
0     11224              1  SKU_643    NaN        1
1     11972              2   SKU839    NaN      one
2      4790              1  SKU-314    NaN        2
3      9893              1  SKU1138    NaN        2
4     13263              2      NAN    NaN        1


/tmp/ipython-input-4041100738.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dirty_report = pd.concat([


In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, initcap, dayofmonth, month, year, quarter, dayofweek, row_number
from pyspark.sql.window import Window
import shutil, glob, os

from graphviz import Digraph

spark = SparkSession.builder.appName("EXL_Round2_ETL").getOrCreate()

users = spark.read.csv("users_cleaned.csv", header=True, inferSchema=True)
products = spark.read.csv("products_cleaned.csv", header=True, inferSchema=True)
orders = spark.read.csv("orders_cleaned.csv", header=True, inferSchema=True)
order_items = spark.read.csv("order_items_cleaned.csv", header=True, inferSchema=True)

dim_users = users.dropDuplicates(['user_id']) \
    .withColumn('name', upper(col('name'))) \
    .withColumn('city', initcap(col('city')))

dim_products = products.dropDuplicates(['product_id']) \
    .withColumn('sku', upper(col('sku'))) \
    .withColumn('product_name', initcap(col('product_name'))) \
    .withColumn('category', initcap(col('category')))

orders = orders.withColumn('order_timestamp', col('order_timestamp').cast('timestamp'))
dim_date = orders.select('order_timestamp').dropDuplicates() \
    .withColumn('day', dayofmonth(col('order_timestamp'))) \
    .withColumn('month', month(col('order_timestamp'))) \
    .withColumn('year', year(col('order_timestamp'))) \
    .withColumn('quarter', quarter(col('order_timestamp'))) \
    .withColumn('weekday', dayofweek(col('order_timestamp')))

window_spec = Window.orderBy('order_timestamp')
dim_date = dim_date.withColumn('date_id', row_number().over(window_spec))

orders = orders.join(dim_date.select('order_timestamp', 'date_id'), on='order_timestamp', how='left')

fact_orders = order_items.join(
    orders.select('order_id','user_id','date_id'), on='order_id', how='left'
).join(
    dim_products.select('product_id','sku'), on='sku', how='left'
)

fact_orders = fact_orders.withColumn('total_amount', col('quantity') * col('price'))

fact_orders = fact_orders.select(
    'order_id','order_item_id','user_id','product_id','date_id','quantity','price','total_amount'
)


def save_single_csv(df, folder_name, output_name):
    df.coalesce(1).write.csv(folder_name, header=True, mode='overwrite')
    part_file = glob.glob(f"{folder_name}/part*.csv")[0]
    shutil.move(part_file, output_name)
    shutil.rmtree(folder_name)

save_single_csv(dim_users, "dim_users_temp", "dim_users.csv")
save_single_csv(dim_products, "dim_products_temp", "dim_products.csv")
save_single_csv(dim_date, "dim_date_temp", "dim_date.csv")
save_single_csv(fact_orders, "fact_orders_temp", "fact_orders.csv")


print("dim_users preview:")
dim_users.show(5)

print("dim_products preview:")
dim_products.show(5)

print("dim_date preview:")
dim_date.show(5)

print("fact_orders preview:")
fact_orders.show(5)


dot = Digraph(comment='Star Schema - EXL Round 2', format='png')
dot.attr(rankdir='LR', size='10')

dot.node('dim_users', 'dim_users\n(user_id, name, city)', shape='box', style='filled', color='lightblue')
dot.node('dim_products', 'dim_products\n(product_id, sku, product_name, category)', shape='box', style='filled', color='lightgreen')
dot.node('dim_date', 'dim_date\n(date_id, day, month, year, quarter, weekday)', shape='box', style='filled', color='lightpink')

dot.node('fact_orders', 'fact_orders\n(order_id, order_item_id, user_id, product_id, date_id, quantity, price, total_amount)', shape='ellipse', style='filled', color='lightyellow')

dot.edge('dim_users', 'fact_orders', label='user_id → user_id')
dot.edge('dim_products', 'fact_orders', label='product_id → product_id')
dot.edge('dim_date', 'fact_orders', label='date_id → date_id')

dot.render('star_schema_diagram', view=True)
print("✅ Star Schema Diagram Generated Successfully!")

dim_users preview:
+-------+-------------+---------+
|user_id|         name|     city|
+-------+-------------+---------+
|      1|      UNKNOWN|  Jodhpur|
|      2|OLIVIA SHARMA|     Pune|
|      3|ANAYA  KAPOOR|Hyderabad|
|      4|      UNKNOWN|  Gwalior|
|      5|      UNKNOWN| Amritsar|
+-------+-------------+---------+
only showing top 5 rows

dim_products preview:
+----------+-----+-------------+-----------+
|product_id|  sku| product_name|   category|
+----------+-----+-------------+-----------+
|         1| SKU1|        Mouse|Electronics|
|         2|  NAN|        Watch|Electronics|
|         3| SKU3|Graphics Card|Electronics|
|         4|SKU-4|         Sofa|  Furniture|
|         5|  NAN|   Television|Electronics|
+----------+-----+-------------+-----------+
only showing top 5 rows

dim_date preview:
+-------------------+---+-----+----+-------+-------+-------+
|    order_timestamp|day|month|year|quarter|weekday|date_id|
+-------------------+---+-----+----+-------+-------+------

In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan, count, when, sum as _sum
from pyspark.sql.types import NumericType

spark = SparkSession.builder.appName("EXL_Round2_DQ").getOrCreate()

dim_users = spark.read.csv("dim_users.csv", header=True, inferSchema=True)
dim_products = spark.read.csv("dim_products.csv", header=True, inferSchema=True)
dim_date = spark.read.csv("dim_date.csv", header=True, inferSchema=True)
fact_orders = spark.read.csv("fact_orders.csv", header=True, inferSchema=True)

def null_check(df, table_name):
    """
    Check nulls in a DataFrame.
    For numeric columns, check nulls + NaNs.
    For non-numeric (string, timestamp), check nulls only.
    """
    print(f"\nNull Value Check - {table_name}:")
    checks = []
    for c in df.columns:
        dtype = df.schema[c].dataType
        if isinstance(dtype, NumericType):
            # count nulls and NaNs for numeric columns
            cond = col(c).isNull() | isnan(col(c))
        else:
            # check nulls only for non-numeric columns
            cond = col(c).isNull()
        # sum up 1 for each row where condition is true
        checks.append(_sum(when(cond, 1).otherwise(0)).alias(c))
    df.select(checks).show(truncate=False)

def duplicate_check(df, table_name, subset_cols=None):
    """
    Check duplicate rows in DataFrame.
    """
    if subset_cols is None:
        subset_cols = df.columns
    dup_count = df.groupBy(subset_cols).count().filter(col("count") > 1).count()
    print(f"\nDuplicate Check - {table_name}: {dup_count} duplicate rows found based on {subset_cols}")

def numeric_check(df, table_name):
    """
    Ensure numeric columns are valid numbers (not null and not NaN).
    """
    print(f"\nNumeric Field Check - {table_name}:")
    for field in df.schema:
        if isinstance(field.dataType, NumericType):
            c = field.name
            invalid_count = df.filter(col(c).isNull() | isnan(col(c))).count()
            print(f"Column {c}: {invalid_count} invalid/null/NaN values")

tables = {
    "dim_users": dim_users,
    "dim_products": dim_products,
    "dim_date": dim_date,
    "fact_orders": fact_orders
}

for name, df in tables.items():
    null_check(df, name)
    duplicate_check(df, name)
    numeric_check(df, name)

def foreign_key_check(fact_df, dim_df, fact_col, dim_col, fact_table_name, dim_table_name):
    missing_count = fact_df.join(dim_df, fact_df[fact_col] == dim_df[dim_col], "left_anti").count()
    print(f"\nForeign Key Check: {fact_col} in {fact_table_name} referencing {dim_col} in {dim_table_name} -> {missing_count} missing values")

foreign_key_check(fact_orders, dim_users, "user_id", "user_id", "fact_orders", "dim_users")
foreign_key_check(fact_orders, dim_products, "product_id", "product_id", "fact_orders", "dim_products")
foreign_key_check(fact_orders, dim_date, "date_id", "date_id", "fact_orders", "dim_date")

def timestamp_check(df, col_name, table_name):
    invalid_count = df.filter(col(col_name).isNull()).count()
    print(f"\nTimestamp Check - {table_name}: {invalid_count} invalid/missing values in {col_name}")

timestamp_check(dim_date, "order_timestamp", "dim_date")

print("\n✅ Data Quality Checks Completed Successfully!")



Null Value Check - dim_users:
+-------+----+----+
|user_id|name|city|
+-------+----+----+
|0      |0   |0   |
+-------+----+----+


Duplicate Check - dim_users: 0 duplicate rows found based on ['user_id', 'name', 'city']

Numeric Field Check - dim_users:
Column user_id: 0 invalid/null/NaN values

Null Value Check - dim_products:
+----------+---+------------+--------+
|product_id|sku|product_name|category|
+----------+---+------------+--------+
|0         |0  |0           |0       |
+----------+---+------------+--------+


Duplicate Check - dim_products: 0 duplicate rows found based on ['product_id', 'sku', 'product_name', 'category']

Numeric Field Check - dim_products:
Column product_id: 0 invalid/null/NaN values

Null Value Check - dim_date:
+---------------+---+-----+----+-------+-------+-------+
|order_timestamp|day|month|year|quarter|weekday|date_id|
+---------------+---+-----+----+-------+-------+-------+
|0              |0  |0    |0   |0      |0      |0      |
+---------------+

In [ ]:

dim_users = spark.read.csv("dim_users.csv", header=True, inferSchema=True)
dim_products = spark.read.csv("dim_products.csv", header=True, inferSchema=True)
dim_date = spark.read.csv("dim_date.csv", header=True, inferSchema=True)
fact_orders = spark.read.csv("fact_orders.csv", header=True, inferSchema=True)

dim_users.createOrReplaceTempView("dim_users")
dim_products.createOrReplaceTempView("dim_products")
dim_date.createOrReplaceTempView("dim_date")
fact_orders.createOrReplaceTempView("fact_orders")

query1 = """
SELECT d.month, SUM(f.total_amount) as monthly_sales
FROM fact_orders f
JOIN dim_date d ON f.date_id = d.date_id
GROUP BY d.month
ORDER BY d.month
"""
result1 = spark.sql(query1)
print("Total Sales per Month:")
result1.show()

query2 = """
SELECT f.product_id, p.product_name, SUM(f.total_amount) as revenue
FROM fact_orders f
JOIN dim_products p ON f.product_id = p.product_id
GROUP BY f.product_id, p.product_name
ORDER BY revenue DESC
LIMIT 10
"""
result2 = spark.sql(query2)
print("Top 10 Products by Revenue:")
result2.show()

query3 = """
SELECT f.user_id, u.name, COUNT(f.order_id) as total_orders
FROM fact_orders f
JOIN dim_users u ON f.user_id = u.user_id
GROUP BY f.user_id, u.name
ORDER BY total_orders DESC
LIMIT 10
"""
result3 = spark.sql(query3)
print("Top 10 Active Users:")
result3.show()

query4 = """
SELECT f.user_id, u.name, AVG(f.total_amount) as avg_order_value
FROM fact_orders f
JOIN dim_users u ON f.user_id = u.user_id
GROUP BY f.user_id, u.name
ORDER BY avg_order_value DESC
LIMIT 10
"""
result4 = spark.sql(query4)
print("Average Order Value per User:")
result4.show()

query5 = """
SELECT p.category, SUM(f.total_amount) as category_revenue
FROM fact_orders f
JOIN dim_products p ON f.product_id = p.product_id
GROUP BY p.category
ORDER BY category_revenue DESC
"""
result5 = spark.sql(query5)
print("Category-wise Revenue:")
result5.show()


query6 = """
SELECT d.quarter, COUNT(f.order_id) as total_orders
FROM fact_orders f
JOIN dim_date d ON f.date_id = d.date_id
GROUP BY d.quarter
ORDER BY d.quarter
"""
result6 = spark.sql(query6)
print("Orders per Quarter:")
result6.show()

Total Sales per Month:
+-----+--------------------+
|month|       monthly_sales|
+-----+--------------------+
|    1|  7582965.2700000005|
|    2|   6592999.879999986|
|    3|  1.30452983300003E7|
|    4|   9811443.680000033|
|    5|   7069406.229999998|
|    6|   5646477.389999993|
|    7|   6103045.850000035|
|    8|    8407933.05000007|
|    9|1.3125284360000074E7|
|   10|  1.18636802800003E7|
|   11|   6285982.710000001|
|   12|   5391534.909999998|
+-----+--------------------+

Top 10 Products by Revenue:
+----------+-------------+-----------------+
|product_id| product_name|          revenue|
+----------+-------------+-----------------+
|       340|    Pen Drive|7433161.630000029|
|       575|   Smartwatch|7433161.630000029|
|       267|    Hard Disk|7433161.630000029|
|       120|         Sofa|7433161.630000029|
|       428|          Bag|7433161.630000029|
|       254|     Notebook|7433161.630000029|
|        74|    Microwave|7433161.630000029|
|       748| Mobile Phone|7433161.